# Solution C — APIM → Event Hub → Blob (Capture) 測試

**目的**：驗證 Solution C 完整 body 落地能力，並與 Solution ① 雙軌獨立。

**隔離原則**：
- 只有送出 header `X-Logging-Channel: solution-c` 的請求才會觸發 EH 寫入
- Solution ①（AppInsights）對所有流量持續啟用，互不干擾
- 每筆請求帶 `X-Run-Id`（=correlationId），用來事後在 EH/Blob/ADX 拼回完整對話

**前置**：
1. `.\scripts\deploy-eventhub-logging.ps1` ✅
2. `.\scripts\apply-eventhub-policy.ps1 -ApiName kunlenewfoundry01` ✅

In [ ]:
import os, time, json, subprocess
from openai import OpenAI
from getpass import getpass

# 認證重點：
# 1. product-scoped subscription key（非 master key），例如 kunlenewfoundry01-proj-default-ai-wl55p1p5w0
# 2. 此 API 用 header 名稱 `api-key`（不是 Ocp-Apim-Subscription-Key）
# 3. 用 OpenAI v1 client（Foundry endpoint 已經是 /openai/v1 路徑）

APIM_BASE_URL     = 'https://testaigw01.azure-api.net/kunlenewfoundry01/openai/v1/'
APIM_SUBSCRIPTION = os.environ.get('APIM_SUBSCRIPTION_KEY') or getpass('APIM Subscription Key (product-scoped, e.g. kunlenewfoundry01-proj-default-ai-wl55p1p5w0): ')
DEPLOYMENT_NAME   = 'Kimi-K2.5'

RUN_ID = f'solc-{int(time.time())}'
print('RUN_ID =', RUN_ID)

client = OpenAI(
    base_url        = APIM_BASE_URL,
    api_key         = 'unused',
    default_headers = {
        'api-key':           APIM_SUBSCRIPTION,
        'X-Logging-Channel': 'solution-c',
        'X-Run-Id':          RUN_ID,
    },
)
# ----- 驗證 helper：從 Blob Capture 拉對應 marker 的事件 -----
import subprocess, io, time as _t, sys, shutil
AZ = shutil.which('az') or shutil.which('az.cmd') or 'az'
RG = 'newfoundry01'
_STORAGE_CACHE = {}

def _get_storage_account():
    if 'name' in _STORAGE_CACHE: return _STORAGE_CACHE['name']
    out = subprocess.run([AZ,'storage','account','list','-g',RG,'-o','json'], capture_output=True, text=True)
    name = [s['name'] for s in json.loads(out.stdout) if s['name'].startswith('staigwc')][0]
    _STORAGE_CACHE['name'] = name
    return name

def _list_blobs(since_iso=None):
    sa = _get_storage_account()
    out = subprocess.run([AZ,'storage','blob','list','--account-name',sa,'--container-name','capture','--auth-mode','login','--num-results','500','-o','json'], capture_output=True, text=True)
    if out.returncode != 0: return []
    blobs = json.loads(out.stdout)
    blobs = [b for b in blobs if int(b['properties']['contentLength']) > 1000]
    if since_iso:
        blobs = [b for b in blobs if b['properties']['lastModified'] >= since_iso]
    return sorted(blobs, key=lambda b: b['properties']['lastModified'], reverse=True)

def _download_blob(name):
    import tempfile, os as _os
    sa = _get_storage_account()
    fd, path = tempfile.mkstemp(suffix='.avro'); _os.close(fd); _os.remove(path)
    subprocess.run([AZ,'storage','blob','download','--account-name',sa,'--container-name','capture','--name',name,'--auth-mode','login','--no-progress','-f',path], capture_output=True)
    with open(path,'rb') as f: raw = f.read()
    try: _os.remove(path)
    except: pass
    return raw

def _parse_avro(raw):
    import fastavro
    events = []
    for rec in fastavro.reader(io.BytesIO(raw)):
        b = rec.get('Body')
        if isinstance(b,(bytes,bytearray)): b = b.decode('utf-8','replace')
        try: events.append(json.loads(b))
        except: events.append({'_raw': b})
    return events

def verify_marker(marker, expect_found=True, max_wait_sec=180, poll_sec=15, since_iso=None):
    """輪詢 blob capture，找到包含 marker 的事件後印出（input body + output body + token usage）。
    expect_found=False: 用於 TC-C4 對照組，預期找不到。"""
    print(f'[verify] marker = {marker}')
    print(f'[verify] storage = {_get_storage_account()} / capture')
    print(f'[verify] expect_found = {expect_found}, max_wait = {max_wait_sec}s')
    deadline = _t.time() + max_wait_sec
    matching = []
    matched_blob = None
    while _t.time() < deadline:
        blobs = _list_blobs(since_iso=since_iso)
        for b in blobs:
            raw = _download_blob(b['name'])
            evts = _parse_avro(raw)
            hits = [e for e in evts if marker in json.dumps(e, ensure_ascii=False)]
            if hits:
                matching = hits; matched_blob = b['name']; break
        if matching: break
        if not expect_found and _t.time() - (deadline - max_wait_sec) > 60:
            # 對照組：等 60s 仍找不到就視為通過
            break
        remain = int(deadline - _t.time())
        print(f'  ...not yet, sleeping {poll_sec}s (remain {remain}s)')
        _t.sleep(poll_sec)
    if expect_found:
        if not matching:
            print(f'❌ FAIL: marker {marker} 在 {max_wait_sec}s 內找不到於 capture container')
            return None
        print(f'✅ found {len(matching)} event(s) in blob: {matched_blob}\n')
        for e in sorted(matching, key=lambda x: {'summary':0,'request-body':1,'response-body':2}.get(x.get('kind'),9)):
            kind = e.get('kind','?')
            print(f'--- {kind} (correlationId={e.get("correlationId")}) ---')
            if kind == 'summary':
                for k in ('method','url','status','durationMs','requestLength','responseLength','requestChunks','responseChunks','subscriptionId','clientIp'):
                    print(f'  {k}: {e.get(k)}')
            elif kind == 'request-body':
                p = e.get('payload','')
                print(f'  chunk {e.get("chunkIndex")}/{e.get("chunkTotal")}, payloadLen={len(p)}')
                try:
                    pj = json.loads(p)
                    print(f'  model: {pj.get("model")}, stream: {pj.get("stream")}, max_tokens: {pj.get("max_tokens")}')
                    msgs = pj.get('messages',[])
                    for m in msgs:
                        print(f'  [{m.get("role")}] {(m.get("content") or "")[:300]}')
                except: print('  (raw):', p[:400])
            elif kind == 'response-body':
                p = e.get('payload','')
                print(f'  chunk {e.get("chunkIndex")}/{e.get("chunkTotal")}, payloadLen={len(p)}')
                if p.lstrip().startswith('data:'):
                    # streaming SSE — 拼回完整 content
                    content_acc = []; usage = None
                    for line in p.splitlines():
                        if not line.startswith('data:'): continue
                        data = line[5:].strip()
                        if data == '[DONE]' or not data: continue
                        try:
                            d = json.loads(data)
                            for ch in d.get('choices',[]) or []:
                                delta = ch.get('delta',{})
                                if delta.get('content'): content_acc.append(delta['content'])
                            if d.get('usage'): usage = d['usage']
                        except: pass
                    full = ''.join(content_acc)
                    print(f'  [streaming] reassembled content len={len(full)} chars')
                    print(f'  preview: {full[:400]}')
                    print(f'  usage (from final SSE chunk): {usage}')
                else:
                    try:
                        pj = json.loads(p)
                        msg = pj['choices'][0].get('message',{})
                        content = msg.get('content') or msg.get('reasoning_content') or ''
                        print(f'  model: {pj.get("model")}, finish_reason: {pj["choices"][0].get("finish_reason")}')
                        print(f'  content len={len(content)} chars, preview:')
                        print(f'  {content[:400]}')
                        print(f'  usage: {pj.get("usage")}   ← ✅ 完整 prompt/completion/total tokens')
                    except Exception as ex:
                        print(f'  (parse-failed: {ex}) raw[0:300]:', p[:300])
            print()
        return matching
    else:
        if matching:
            print(f'❌ FAIL: 對照組 marker {marker} **不應**出現在 EH，卻找到 {len(matching)} 筆於 {matched_blob}')
            return matching
        print(f'✅ PASS: 對照組 marker {marker} 確實**未**出現在 EH（隔離有效）')
        return []

# 記錄測試起始時間，避免 verify 時掃到很舊的 blob
RUN_START_ISO = _t.strftime('%Y-%m-%dT%H:%M:%SZ', _t.gmtime(_t.time() - 60))
print('RUN_START_ISO:', RUN_START_ISO)

## TC-C1 — Non-streaming 短回覆（驗 happy path）

In [ ]:
marker = f'{RUN_ID}-c1'
resp = client.chat.completions.create(
    model = DEPLOYMENT_NAME,
    messages = [{'role':'user','content': f'[{marker}] 用一句話說明 Event Hub'}],
    max_tokens = 200,
    extra_headers = {'X-Run-Id': marker},
)
print('Marker :', marker)
print('Status :', 'OK')
print('Usage  :', resp.usage)
_msg = resp.choices[0].message
_text = _msg.content or getattr(_msg, 'reasoning_content', None) or ''
print('Content:', _text[:200])

### 🔍 驗證 TC-C1：等 Capture flush 後從 Blob 拉回 — 確認 input message + output content + **完整 token usage**

In [ ]:
verify_marker(marker, expect_found=True, max_wait_sec=180, since_iso=RUN_START_ISO)

## TC-C2 — 大型回覆（驗證超過 256KB 不被截斷，多 chunk）

In [ ]:
marker = f'{RUN_ID}-c2'
resp = client.chat.completions.create(
    model = DEPLOYMENT_NAME,
    messages = [{'role':'user','content': f'[{marker}] 用 reasoning 詳細解釋 Kubernetes 從零到生產所有概念，至少 8000 字繁中'}],
    max_tokens = 8000,
    extra_headers = {'X-Run-Id': marker},
)
print('Marker     :', marker)
_msg = resp.choices[0].message
_text = _msg.content or getattr(_msg, 'reasoning_content', None) or ''
print('Resp bytes :', len(_text.encode('utf-8')))
print('Usage      :', resp.usage)

### 🔍 驗證 TC-C2：大型回覆 — 確認 multi-chunk reassembly + **token usage 完整**

In [ ]:
verify_marker(marker, expect_found=True, max_wait_sec=180, since_iso=RUN_START_ISO)

## TC-C3 — Streaming SSE（驗證 R4 場景下方案 C 是否拿得到完整 body）

In [ ]:
marker = f'{RUN_ID}-c3'
stream = client.chat.completions.create(
    model = DEPLOYMENT_NAME,
    messages = [{'role':'user','content': f'[{marker}] streaming 解釋 reasoning model 如何推理'}],
    max_tokens = 4000,
    stream = True,
    stream_options = {'include_usage': True},
    extra_headers = {'X-Run-Id': marker},
)
chars = 0
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        chars += len(chunk.choices[0].delta.content)
print('Marker:', marker)
print('Total content chars:', chars)

### 🔍 驗證 TC-C3：Streaming — 確認 SSE 完整捕捉 + 從最後一筆 chunk 取出 usage

In [ ]:
verify_marker(marker, expect_found=True, max_wait_sec=180, since_iso=RUN_START_ISO)

## TC-C4 — 對照組：**不帶** header（應只進 Solution ①，**不**進 EH）

In [ ]:
# 對照組：故意不帶 X-Logging-Channel header → Solution C policy 應 skip 寫 EH
marker = f'{RUN_ID}-c4-controlgroup'
control = OpenAI(
    base_url        = APIM_BASE_URL,
    api_key         = 'unused',
    default_headers = {'api-key': APIM_SUBSCRIPTION},  # 故意不帶 X-Logging-Channel
)
resp = control.chat.completions.create(
    model = DEPLOYMENT_NAME,
    messages = [{'role':'user','content': f'[{marker}] hello'}],
    max_tokens = 50,
)
print('Marker:', marker)
print('Status: OK')
print('預期：EH/Blob 應**找不到**這個 marker；只能在 Solution ① 的 AppInsights 找到')

### 🔍 驗證 TC-C4 對照組：marker **不應**出現在 EH（隔離驗證）

In [ ]:
verify_marker(marker, expect_found=False, max_wait_sec=180, since_iso=RUN_START_ISO)

## ✅ 驗收標準

| 項目 | 預期 |
|---|---|
| TC-C1 (短) | EH 內找得到 1 summary + 1 request-body chunk + 1 response-body chunk |
| TC-C2 (大) | response-body 拆 ≥ 2 chunk，總 length 大於 256KB（Solution ① 在此會被截）|
| TC-C3 (stream) | 仍能拿到完整 body（驗證 R4 在 Capture 路徑下表現）|
| TC-C4 (對照) | EH/Blob 內**沒有** `c4-controlgroup` 字樣 → 證明 header 隔離有效 |
| Solution ① | 全部 4 個 marker 都應在 AppInsights AppDependencies 找得到 → 證明雙軌獨立 |

## 後續查詢進階

如要長期 ad-hoc 查詢，建議掛 ADX external table 指向 capture container（範例：`kql/queries-eventhub.kql` C1-C5）。